# Chapter 12 — Dynamic Programming

**Source:** Wengrow, *A Common-Sense Guide to Data Structures and Algorithms*, Ch. 12

## The big idea
Recursion makes code elegant, but it can silently become **O(2^N)**. The cause is almost always **the same call being made again and again**. This chapter gives you three tools to fix that:

| Problem | Fix |
|---|---|
| Same call written twice in the code | Save the result in a **variable** |
| Overlapping subproblems (e.g. Fibonacci) | **Memoization** (a hash table remembers results) |
| Recursion is awkward or heavy | **Bottom-up** (a loop instead of recursion) |

**Dynamic programming** = optimizing a recursive problem that has overlapping subproblems, so that no call is computed twice.

## 1. Unnecessary recursive calls — the `max` example

The function compares `array[0]` with the max of the rest of the array.
The mistake is writing `max(array[1:])` **twice**, once in the `if` and once in the `else`.
Every mention is a full avalanche of recursive calls.

Calls made, by array size N:

| N | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| calls | 1 | 3 | 7 | 15 | 31 |

Add one element and the calls roughly **double** → **O(2^N)**.

**The fix:** call it once and store the result in a variable.
The calls drop to N → **O(N)**.

Takeaway: *a tiny change (saving a computation in a variable) took the function from O(2^N) to O(N).*

In [ ]:
# --- BAD: O(2^N) ---
calls = 0

def max_bad(array):
    global calls
    calls += 1
    if len(array) == 1:
        return array[0]
    if array[0] > max_bad(array[1:]):      # call #1 on the remainder
        return array[0]
    else:
        return max_bad(array[1:])          # call #2 on the SAME remainder

calls = 0
print(max_bad([1, 2, 3, 4]), "calls:", calls)   # 15 calls


# --- GOOD: O(N) ---
def max_good(array):
    global calls
    calls += 1
    if len(array) == 1:
        return array[0]
    max_of_remainder = max_good(array[1:])      # called ONCE, result saved
    if array[0] > max_of_remainder:
        return array[0]
    else:
        return max_of_remainder

calls = 0
print(max_good([1, 2, 3, 4]), "calls:", calls)  # 4 calls

## 2. Overlapping subproblems — Fibonacci

```python
def fib(n):
    if n == 0 or n == 1:
        return n
    return fib(n - 2) + fib(n - 1)
```

This time you can't just save one variable. `fib(n-2)` and `fib(n-1)` are **both** needed, and they are different calls.
But they repeat each other's work: `fib(5)` calls `fib(3)` and `fib(4)`, and `fib(4)` calls `fib(3)` **again**.

> **Subproblem** = a smaller version of the same problem.
> **Overlapping** = different branches of the recursion need the *same* subproblem.

Result: **O(2^N)**. This is the same pattern as your staircase function `number_of_paths`.

In [ ]:
calls = 0

def fib_naive(n):
    global calls
    calls += 1
    if n == 0 or n == 1:
        return n
    return fib_naive(n - 2) + fib_naive(n - 1)

for n in [5, 10, 20, 25]:
    calls = 0
    fib_naive(n)
    print(f"fib({n}) -> {calls:>7} calls")

## 3. Technique 1: Memoization

**Memoization** = remember results you already computed.
Keep a hash table (a Python `dict`), `{n: fib(n)}`, and **check it before recursing**.

How the dict is shared: pass it as a **second parameter**.
A dict is one object in memory, so every recursive call sees and fills the same one.

Calls for memoized `fib`: **2N − 1** → **O(N)**.

| N | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| calls | 1 | 3 | 5 | 7 | 9 | 11 |

**Python gotchas**
- The book writes `if not memo.get(n)`. That breaks if a stored value is `0` (falsy). Use `if n not in memo`.
- `def fib(n, memo={})` is *dangerous* in Python: the default dict is created once and shared between every call to the function. It happens to be harmless for Fibonacci (the answers never change), but it is a classic bug, so `memo=None` is safer.
- `@functools.lru_cache` and your own `@memoize` from Week 2 do exactly this automatically.

In [ ]:
calls = 0

def fib_memo(n, memo=None):
    global calls
    calls += 1
    if memo is None:
        memo = {}
    if n == 0 or n == 1:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 2, memo) + fib_memo(n - 1, memo)
    return memo[n]

for n in [5, 10, 20, 25]:
    calls = 0
    fib_memo(n)
    print(f"fib({n}) -> {calls:>3} calls")

# The same idea with your decorator from Week 2:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib_lru(n):
    if n < 2:
        return n
    return fib_lru(n - 2) + fib_lru(n - 1)

print(fib_lru(100))   # instant, would never finish naively

## 4. Technique 2: Going bottom-up

**Bottom-up** = drop recursion and build the answer with a loop, starting from the base cases.
It counts as dynamic programming because it also avoids duplicate calls.

For Fibonacci you only need the last two numbers, so keep two variables.
That's N loop steps → **O(N)**.

It also uses no call stack and no hash table, so it uses less memory.

## 5. Memoization vs. bottom-up

| | Memoization | Bottom-up |
|---|---|---|
| Keeps recursion? | Yes | No (loop) |
| Extra memory | call stack + hash table | little or none |
| Best when | recursion is the *natural, intuitive* solution (e.g. staircase) | the loop is just as easy (e.g. Fibonacci) |

Rule of thumb from the book: **go bottom-up unless the recursive solution is much more intuitive**. In that case keep the recursion and add memoization.
Python has a practical reason too: the default recursion limit is about 1000, so deep recursion can crash with `RecursionError`.

In [ ]:
def fib_bottom_up(n):
    if n == 0:
        return 0
    a, b = 0, 1                      # fib(0), fib(1)
    for _ in range(1, n):
        a, b = b, a + b              # book uses a temp variable; Python can swap in one line
    return b

print([fib_bottom_up(i) for i in range(11)])   # 0, 1, 1, 2, 3, 5, 8 ...
print(fib_bottom_up(100))

## Summary

- Extra recursive calls are the main reason recursion gets slow.
- **Step 1:** is the *same* call written more than once in your code? Save it in a variable.
- **Step 2:** do different branches need the same subproblem? That's *overlapping subproblems*. Use **memoization** (pass a `dict` along) or go **bottom-up** (a loop).
- Both techniques turn **O(2^N) into O(N)**.
- Chapter 13 uses recursion for speed (Quicksort), so recursion needs to be second nature before it.

## Exercises (book solutions: p. 450)

### Exercise 1: `add_until_100`
**Input:** a list of numbers, e.g. `[10, 20, 30, 40]`.
**What it does:** adds the numbers up, but **skips** any number that would push the running total above 100.
**Your job:** the function below calls `add_until_100(array[1:])` up to three times per level. Rewrite it so it's called **once** (save it in a variable).

### Exercise 2: `golomb`
**Input:** one integer `n >= 1`: which position of the Golomb sequence you want.
**Your job:** use memoization (a `memo` dict as a second parameter). You don't need to understand the sequence itself. Check: `golomb(1)` through `golomb(10)` should give `1, 2, 2, 3, 3, 4, 4, 4, 5, 5`.

### Exercise 3: `unique_paths`
**Input:** two integers, `rows` and `columns`: the size of a grid.
**What it computes:** how many different routes lead from the top-left cell to the bottom-right cell if you can only move **right** or **down**. (This is the same Unique Paths exercise from Chapter 11.)
**Your job:** add memoization. Careful: the answer depends on **two** numbers, so the dict key needs to hold both `(rows, columns)`.

In [ ]:
# Exercise 1 — fix the unnecessary recursion
def add_until_100(array):
    if len(array) == 0:
        return 0
    if array[0] + add_until_100(array[1:]) > 100:
        return add_until_100(array[1:])
    else:
        return array[0] + add_until_100(array[1:])

# Inputs to try:
print(add_until_100([10, 20, 30, 40]))     # nothing exceeds 100
print(add_until_100([50, 40, 30, 20, 10])) # something gets skipped
print(add_until_100([]))                   # base case
print(add_until_100([60] * 10))            # slow before the fix? count the calls!

In [ ]:
# Exercise 2 — add memoization
def golomb(n):
    if n == 1:
        return 1
    return 1 + golomb(n - golomb(golomb(n - 1)))

# Inputs to try (watch how slow the larger ones get BEFORE you fix it):
print([golomb(i) for i in range(1, 11)])   # expect 1, 2, 2, 3, 3, 4, 4, 4, 5, 5
# print(golomb(35))                        # try it before and after memoizing

In [ ]:
# Exercise 3 — add memoization
def unique_paths(rows, columns):
    if rows == 1 or columns == 1:
        return 1
    return unique_paths(rows - 1, columns) + unique_paths(rows, columns - 1)

# Inputs to try:
print(unique_paths(3, 3))      # small grid
print(unique_paths(3, 7))      # medium grid
# print(unique_paths(18, 18))  # try before and after memoizing